In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN observation_concept_id = 0 THEN 1 ELSE 0 END) AS zero_observation_concept_rows,
  ROUND(100.0 * SUM(CASE WHEN observation_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_zero_observation_concept_id
FROM _exponent.omop_tw.observation;


In [0]:
%sql
SELECT observation_concept_id, COUNT(*) AS rows
FROM _exponent.omop_tw.observation
GROUP BY 1
ORDER BY rows DESC;


In [0]:
%sql
SELECT COUNT(*) AS current_observation_input_rows
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_item_result ir
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_result r
  ON ir.CurrentID = r.ID
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept meas_concept
  ON meas_concept.source_id = ir.QODE
 AND meas_concept.source_table = 'dbo_qo_de'
 AND meas_concept.domain_id = 'Measurement'
 AND meas_concept.source_system = 'allscripts_tw'
WHERE ir.ID IS NOT NULL
  AND ir.PatientID IS NOT NULL
  AND r.NumericResult IS NULL
  AND meas_concept.omop_concept_id IS NULL
  AND CAST(COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS DATE) >= '1950-01-01';


In [0]:
%sql
SELECT
  ir.QODE,
  q.EntryName,
  COUNT(*) AS rows
FROM _exponent._bronze_allscripts_tw_works_vw.dbo_item_result ir
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_result r
  ON ir.CurrentID = r.ID
LEFT JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_de q
  ON q.ID = ir.QODE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept obs_concept
  ON obs_concept.source_id = ir.QODE
 AND obs_concept.source_table = 'dbo_qo_de'
 AND obs_concept.domain_id = 'Observation'
 AND obs_concept.source_system = 'allscripts_tw'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept meas_concept
  ON meas_concept.source_id = ir.QODE
 AND meas_concept.source_table = 'dbo_qo_de'
 AND meas_concept.domain_id = 'Measurement'
 AND meas_concept.source_system = 'allscripts_tw'
WHERE ir.ID IS NOT NULL
  AND ir.PatientID IS NOT NULL
  AND r.NumericResult IS NULL
  AND meas_concept.omop_concept_id IS NULL
  AND CAST(COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS DATE) >= '1950-01-01'
  AND obs_concept.omop_concept_id IS NULL
GROUP BY 1,2
ORDER BY rows DESC
LIMIT 50;


In [0]:
%sql
SELECT COUNT(DISTINCT source_id) AS promoted_observation_qode_count
FROM _exponent.omop_mapping.domain_source_to_concept
WHERE source_system = 'allscripts_tw'
  AND source_table = 'dbo_qo_de'
  AND domain_id = 'Observation';


In [0]:
%sql
SELECT COUNT(DISTINCT source_id) AS promoted_measurement_qode_count
FROM _exponent.omop_mapping.domain_source_to_concept
WHERE source_system = 'allscripts_tw'
  AND source_table = 'dbo_qo_de'
  AND domain_id = 'Measurement';


In [0]:
%sql
SELECT COUNT(DISTINCT id) AS results_store_qode_count
FROM _exponent.results_store.omop_mapping_qo_de_observation_final_output_v1;


In [0]:
%sql
SELECT
  COUNT(*) AS missing_promotion_rows,
  SUM(CASE WHEN rs.concept_id IS NOT NULL AND rs.concept_id <> '' THEN 1 ELSE 0 END) AS missing_with_concept,
  SUM(CASE WHEN rs.concept_id IS NULL OR rs.concept_id = '' THEN 1 ELSE 0 END) AS missing_without_concept
FROM _exponent.results_store.omop_mapping_qo_de_observation_final_output_v1 rs
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept d
  ON d.source_id = CAST(rs.id AS STRING)
 AND d.source_system = 'allscripts_tw'
 AND d.source_table = 'dbo_qo_de'
 AND d.domain_id = 'Observation'
WHERE d.source_id IS NULL;


In [0]:
%sql
SELECT
  rr.best_domain_id,
  rr.final_status,
  COUNT(*) AS rows
FROM _exponent.results_store.omop_mapping_qo_de_observation_row_results_v1 rr
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept d
  ON d.source_id = CAST(rr.id AS STRING)
 AND d.source_system = 'allscripts_tw'
 AND d.source_table = 'dbo_qo_de'
 AND d.domain_id = 'Observation'
WHERE d.source_id IS NULL
GROUP BY 1,2
ORDER BY rows DESC;


In [0]:
%sql
SELECT
  COUNT(*) AS missing_observation_with_concept
FROM _exponent.results_store.omop_mapping_qo_de_observation_row_results_v1 rr
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept d
  ON d.source_id = CAST(rr.id AS STRING)
 AND d.source_system = 'allscripts_tw'
 AND d.source_table = 'dbo_qo_de'
 AND d.domain_id = 'Observation'
WHERE d.source_id IS NULL
  AND rr.best_domain_id = 'Observation'
  AND rr.concept_id IS NOT NULL
  AND rr.concept_id <> '';


In [0]:
%sql
SELECT
  rr.final_status,
  COUNT(*) AS rows
FROM _exponent.results_store.omop_mapping_qo_de_observation_row_results_v1 rr
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept d
  ON d.source_id = CAST(rr.id AS STRING)
 AND d.source_system = 'allscripts_tw'
 AND d.source_table = 'dbo_qo_de'
 AND d.domain_id = 'Observation'
WHERE d.source_id IS NULL
  AND rr.best_domain_id = 'Observation'
  AND rr.concept_id IS NOT NULL
  AND rr.concept_id <> ''
GROUP BY 1
ORDER BY rows DESC;


In [0]:
%sql
SELECT
  rr.id,
  rr.entryname,
  rr.concept_id,
  rr.best_domain_id,
  rr.final_status
FROM _exponent.results_store.omop_mapping_qo_de_observation_row_results_v1 rr
WHERE rr.id IN ('906', '907', '862', '1052', '5459')
ORDER BY rr.id;


In [0]:
%sql
SELECT concept_id, concept_name, domain_id, vocabulary_id, concept_class_id, standard_concept
FROM _exponent.omop.concept
WHERE concept_id IN (883650, 1990490, 784195, 586570, 2073918)
ORDER BY concept_id;


In [0]:
%sql
SELECT
  rr.id,
  rr.entryname,
  rr.concept_id,
  rr.best_domain_id,
  rr.final_status
FROM _exponent.results_store.omop_mapping_qo_de_measurement_row_results_v1 rr
WHERE rr.id IN ('906', '907', '862', '1052', '5459')
ORDER BY rr.id;


In [0]:
%sql
SELECT source_id, source_value, omop_concept_id, omop_concept_name
FROM _exponent.omop_mapping.domain_source_to_concept
WHERE source_system = 'allscripts_tw'
  AND source_table = 'dbo_qo_de'
  AND domain_id = 'Measurement'
  AND source_id IN (906, 907, 862, 1052, 5459)
ORDER BY source_id;
